# Usage of DeepRaman


Here, part of the liquid and powder mixture dataset is taken as an example to show the process of DeepRaman.

In [1]:
import tensorflow as tf
import numpy as np
from sklearn.linear_model import enet_path
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt 
import tensorflow.keras.backend as K
from tensorflow.keras.layers import Layer
from scipy.sparse import spdiags,eye,csc_matrix, diags
import copy
import csv
from DeepRaman.DeepRaman.training import SpatialPyramidPooling

### Import Data


In [18]:
# Database
datafile0 = r'C:\Users\MathiasCharconnet\PycharmProjects\DeepRaman\DeepRaman\data\database_for_Liquid_and_powder_mixture.npy'
datafile3 = r'C:\Users\MathiasCharconnet\PycharmProjects\DeepRaman/DeepRaman/data/training_Y.npy'
#spectrum_pure = np.load(datafile0)
training = np.load(datafile3)
# Unknow
datafile1 =r'C:\Users\MathiasCharconnet\PycharmProjects\DeepRaman\unknown_Liquid_and_powder_mixture.npy'
spectrum_mix = np.load(datafile1)     
# Component information of database
csv_reader = csv.reader(open(r'C:\Users\MathiasCharconnet\PycharmProjects\DeepRaman\database_for_Liquid_and_powder_mixture.csv', encoding='utf-8'))
DBcoms = [row for row in csv_reader]   
print('Unknow：',spectrum_mix.shape)
print('Database：',DBcoms)

Unknow： (6, 881)
Database： [['Acetonitrile_75-0508_2241'], ['Ethanol_2241'], ['Methanol_2241'], ['Polyacrylamide_2241'], ['Sodium Acetate Trihydrate _2241'], ['Sodium Carbonate_2241']]


In [22]:
training[2]

array([[1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.]])

In [10]:
datafile0 = r'C:\Users\MathiasCharconnet\PycharmProjects\DeepRaman\DeepRaman/data/training_Y.npy'
spectra=np.load(datafile0)
spectra.shape

ValueError: Cannot load file containing pickled data when allow_pickle=False

### Data process

In [23]:
spectrum_pure_sc =  copy.deepcopy(spectrum_pure)
spectrum_mix_sc = copy.deepcopy(spectrum_mix)
for i in range(spectrum_mix.shape[0]):
    spectrum_mix_sc[i,:] = spectrum_mix[i,:]/np.max(spectrum_mix[i,:])
for i in range(spectrum_pure.shape[0]):
    spectrum_pure_sc[i,:] = spectrum_pure[i,:]/np.max(spectrum_pure[i,:])


X = np.zeros((spectrum_mix_sc.shape[0]*spectrum_pure_sc.shape[0],2,881,1))

for p in range(spectrum_mix_sc.shape[0]):
    for q in range(spectrum_pure_sc.shape[0]):
        X[int(p*spectrum_pure_sc.shape[0]+q),0,:,0] = spectrum_mix_sc[p,:]
        X[int(p*spectrum_pure_sc.shape[0]+q),1,:,0] = spectrum_pure_sc[q,:]

### Reload and predict

In [24]:
custom_objects = {
    'SpatialPyramidPooling': SpatialPyramidPooling
}

re_model = tf.keras.models.load_model(r'C:\Users\MathiasCharconnet\PycharmProjects\DeepRaman\DeepRaman/model/model.h5',custom_objects=custom_objects)
y = re_model.predict(X)


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Index'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Index'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


In [28]:
y

array([[0.2877605 , 0.67023814],
       [0.22297662, 0.7342396 ],
       [0.14585248, 0.8132731 ],
       [0.40446982, 0.60173213],
       [0.33591717, 0.63262784],
       [0.67126596, 0.39053312],
       [0.60048497, 0.35565665],
       [0.23378575, 0.6527055 ],
       [0.04320839, 0.9505312 ],
       [0.20420682, 0.7822577 ],
       [0.287808  , 0.6417041 ],
       [0.44956368, 0.5386763 ],
       [0.07504675, 0.8889427 ],
       [0.19619805, 0.752136  ],
       [0.16936627, 0.75183785],
       [0.22007832, 0.7520894 ],
       [0.19341269, 0.7574045 ],
       [0.38147014, 0.6417788 ],
       [0.11410406, 0.8897381 ],
       [0.11857682, 0.8105409 ],
       [0.13135561, 0.83270043],
       [0.01522431, 0.9902927 ],
       [0.02363262, 0.9712906 ],
       [0.03018588, 0.97604406],
       [0.12512895, 0.8940538 ],
       [0.11531454, 0.8295506 ],
       [0.14616379, 0.8509446 ],
       [0.01671281, 0.9909885 ],
       [0.02169853, 0.97795117],
       [0.00735238, 0.9958953 ],
       [0.

### Baseline removal

In [31]:
def WhittakerSmooth(x, lamb, w):
    m=w.shape[0]
    W=spdiags(w,0,m,m)
    D=eye(m-1,m,1)-eye(m-1,m)
    return spsolve((W+lamb*D.transpose()*D),w*x)

def airPLS(x, lamb=10, itermax=10):
    m=x.shape[0]
    w=np.ones(m)
    for i in range(itermax):
        z=WhittakerSmooth(x,lamb,w)
        d=x-z
        if sum(abs(d[d<0]))<0.001*sum(abs(x)):
            break;
        w[d<0]=np.exp(i*d[d<0]/sum(d[d<0]))
        w[d>=0]=0
    return z

def airPLS_MAT(X, lamb=10, itermax=10):
    B=X.copy()
    for i in range(X.shape[0]):
        B[i,]=airPLS(X[i,],lamb,itermax)
    return X-B

def WhittakerSmooth_MAT(X, lamb=1):
    C=X.copy()
    w=np.ones(X.shape[1])
    for i in range(X.shape[0]):
        C[i,]=WhittakerSmooth(X[i,:], lamb, w)

    return C

spectrum_pure = WhittakerSmooth_MAT(spectrum_pure, lamb=1)
spectrum_pure = airPLS_MAT(spectrum_pure, lamb=10, itermax=10)
spectrum_mix = WhittakerSmooth_MAT(spectrum_mix, lamb=1)
spectrum_mix = airPLS_MAT(spectrum_mix, lamb=10, itermax=10)

### NN-EN for ratio estimation

In [32]:
for cc in range(spectrum_mix.shape[0]):
    com=[]
    coms = []
    ra2 = []
    for ss in range(cc*spectrum_pure.shape[0],(cc+1)*spectrum_pure.shape[0]):

        if y[ss,1]>=0.5:
            com.append(ss%spectrum_pure.shape[0])


    X = spectrum_pure[com]
    coms = [DBcoms[com[h]] for h in range(len(com))]

    _, coefs_lasso, _ = enet_path(X.T, spectrum_mix[cc,:], l1_ratio=0.96,
                              positive=True, fit_intercept=False)
    ratio = coefs_lasso[:, -1]
    ratio_sc = copy.deepcopy(ratio)

    for ss2 in range(ratio.shape[0]):
        ratio_sc[ss2]=ratio[ss2]/np.sum(ratio)


    print('The',cc, 'spectra may contain:',coms)
    print('The corresponding ratio is:', ratio_sc)

The 0 spectra may contain: [['Acetonitrile_75-0508_2241'], ['Ethanol_2241'], ['Methanol_2241']]
The corresponding ratio is: [0.19772626 0.31080356 0.49147018]
The 1 spectra may contain: [['Acetonitrile_75-0508_2241'], ['Ethanol_2241'], ['Methanol_2241']]
The corresponding ratio is: [0.0785501  0.21088816 0.71056175]
The 2 spectra may contain: [['Acetonitrile_75-0508_2241'], ['Ethanol_2241'], ['Methanol_2241']]
The corresponding ratio is: [0.30807043 0.30981864 0.38211093]
The 3 spectra may contain: [['Polyacrylamide_2241'], ['Sodium Acetate Trihydrate _2241']]
The corresponding ratio is: [0.74317879 0.25682121]
The 4 spectra may contain: [['Polyacrylamide_2241'], ['Sodium Acetate Trihydrate _2241'], ['Sodium Carbonate_2241']]
The corresponding ratio is: [0.38000182 0.14882521 0.47117298]
The 5 spectra may contain: [['Polyacrylamide_2241'], ['Sodium Acetate Trihydrate _2241'], ['Sodium Carbonate_2241']]
The corresponding ratio is: [0.61626663 0.22985714 0.15387623]
